In [1]:
import k2
import torch
import numpy as np
import pynini
from pynini.lib import pynutil
import io
import os
from glob import glob
import random
from collections import defaultdict
import pickle

In [2]:

# --- 1. Setup CUDA Device ---
# Check if CUDA is available and set the device.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda", 0)
    print(f"CUDA device found: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available. Running on CPU (install CUDA/NVIDIA drivers for GPU acceleration).")

CUDA device found: NVIDIA GeForce RTX 4090


In [3]:
def fst_to_arc_dict(fst_pynini):
    arcs = defaultdict(list)
    aux_labels = defaultdict(list)
    scores = defaultdict(list)
    states = list(fst_pynini.states())
    final_state = max(states) + 1
    for state_id in states:
        for arc in fst_pynini.arcs(state_id):
            arcs[state_id].append([state_id, arc.nextstate, arc.ilabel, 0])
            aux_labels[state_id].append(arc.olabel)
            scores[state_id].append(float(arc.weight))
            if fst_pynini.final(arc.nextstate) != pynini.Weight.zero('tropical'):
                arcs[arc.nextstate].append(
                    [arc.nextstate, final_state, -1, 0]
                )
                aux_labels[arc.nextstate].append(-1)
                scores[arc.nextstate].append(float(fst_pynini.final(arc.nextstate)))
    
    arclist = []
    auxlabellist = []
    scorelist = []
    for state_id in states:
        arclist.extend(arcs[state_id])
        auxlabellist.extend(aux_labels[state_id])
        scorelist.extend(scores[state_id])


    arcs_tensor = torch.tensor(arclist, dtype=torch.int32)
    aux_labels_tensor = torch.tensor(auxlabellist, dtype=torch.int32)
    arc_dict = {
        'arcs': arcs_tensor,
        'aux_labels': aux_labels_tensor,
        'scores': torch.tensor(scorelist, dtype=torch.float32),
    }
    return arc_dict

In [4]:
def k2_fst(fst_pynini):
    arc_dict = fst_to_arc_dict(fst_pynini)
    scores = arc_dict.pop('scores')
    k2_fst_obj = k2.Fsa.from_dict(arc_dict).to(DEVICE)
    k2_fst_obj.scores = scores.to(DEVICE)
    return k2_fst_obj

boy = pynini.cross("cat", "dog") | pynini.cross("mouse", "cheese")
k2_fst(boy)

In [5]:
from src.fst_helpers import fst, get_lattice_strs, extract_word_from_labels
from src.parser import get_main_parser

boy = fst("àpɾí", weight=0.5) | fst("ápɾí", weight=1.5)
lmzr, alzr, inflr = get_main_parser()

get_lattice_strs(boy@lmzr)

['àpɾí[case=unmarked][number=unmarked][aux=unmarked][final_lowering=unmarked][fv=unmarked][left_h=unmarked][part_of_speech=noun]',
 'àpɾí[case=nominative][number=singular][aux=unmarked][final_lowering=unmarked][fv=unmarked][left_h=unmarked][part_of_speech=noun]',
 'àpɾí[case=unmarked][number=unmarked][aux=unmarked][final_lowering=unmarked][fv=unmarked][left_h=true][part_of_speech=noun]',
 'àpɾí[case=nominative][number=singular][aux=unmarked][final_lowering=unmarked][fv=unmarked][left_h=true][part_of_speech=noun]']

In [6]:
boy_k2 = k2_fst(boy)
lmzr_k2 = k2_fst(lmzr)

# k2.compose needs both inputs to be on CPU
boy_k2 = k2.arc_sort(boy_k2)
lmzr_k2 = k2.arc_sort(lmzr_k2)
lattice = k2.compose(boy_k2.to('cpu'), lmzr_k2.to('cpu'))

In [7]:
# unless we pass treat_epsilons_specially=False
boy_k2 = k2.add_epsilon_self_loops(boy_k2)
lmzr_k2 = k2.add_epsilon_self_loops(lmzr_k2)
boy_k2 = k2.arc_sort(boy_k2)
lmzr_k2 = k2.arc_sort(lmzr_k2)
lattice = k2.compose(boy_k2, lmzr_k2, treat_epsilons_specially=False)
lattice = k2.remove_epsilon_self_loops(lattice)

In [8]:
lattice_batch = k2.create_fsa_vec([lattice, lattice])
best_path = k2.shortest_path(lattice_batch, use_double_scores=True)
best_path.labels, best_path.num_arcs, extract_word_from_labels(best_path.labels, strip_eos=False)

(tensor([ 0, 61, 63, 24, 49, 51, 62,  0,  0,  0,  0,  0,  0,  0, -1,  0, 61, 63,
         24, 49, 51, 62,  0,  0,  0,  0,  0,  0,  0, -1], device='cuda:0',
        dtype=torch.int32),
 30,
 'àpɾí<ENDOFSENTENCE>àpɾí<ENDOFSENTENCE>')

In [9]:
from k2.nbest import Nbest
nbest=Nbest.from_lattice(lattice_batch, num_paths=5)
nbest.fsa.labels, extract_word_from_labels(nbest.fsa.labels, strip_eos=False)

(tensor([ 0, 61, 63, 24, 49, 51, 62,  0,  0,  0,  0,  0,  0,  0, -1,  0, 61, 63,
         24, 49, 51, 62,  0,  0,  0,  0,  0,  0,  0, -1,  0, 61, 63, 24, 49, 51,
         62,  0,  0,  0,  0,  0,  0,  0, -1,  0, 61, 63, 24, 49, 51, 62,  0,  0,
          0,  0,  0,  0,  0, -1], device='cuda:0', dtype=torch.int32),
 'àpɾí<ENDOFSENTENCE>àpɾí<ENDOFSENTENCE>àpɾí<ENDOFSENTENCE>àpɾí<ENDOFSENTENCE>')

In [10]:
# from src.search import get_searchable_main_parser
# from src.fst_helpers import get_lattice_strs_and_weights
# left_factor, lexicon = get_searchable_main_parser()

In [11]:
# query = fst("àprí")

# get_lattice_strs_and_weights(query@left_factor@lexicon, nshortest=3)

In [12]:
# left_factor_k2 = k2_fst(left_factor)
# lexicon_k2 = k2_fst(lexicon)
# query_k2 = k2_fst(query)


In [13]:
# torch.save(left_factor_k2.as_dict(), ".cache/left_factor_k2.pt")
# torch.save(lexicon_k2.as_dict(), ".cache/lexicon_k2.pt")
# torch.save(query_k2.as_dict(), ".cache/query_k2.pt")

In [14]:
left_factor_k2_dict = torch.load(".cache/left_factor_k2.pt", weights_only=False)
lexicon_k2_dict = torch.load(".cache/lexicon_k2.pt", weights_only=False)
query_k2_dict = torch.load(".cache/query_k2.pt", weights_only=False)

left_factor_k2 = k2.Fsa.from_dict(left_factor_k2_dict).to(DEVICE)
lexicon_k2 = k2.Fsa.from_dict(lexicon_k2_dict).to(DEVICE)
query_k2 = k2.Fsa.from_dict(query_k2_dict).to(DEVICE)

In [15]:
query_k2 = k2.add_epsilon_self_loops(query_k2)
# left_factor_k2 = k2.add_epsilon_self_loops(left_factor_k2)

query_k2 = k2.arc_sort(query_k2)
left_factor_k2 = k2.arc_sort(left_factor_k2)
lexicon_k2 = k2.arc_sort(lexicon_k2)

lexicon_k2 = k2.top_sort(lexicon_k2)

In [16]:
composed_query = k2.compose(query_k2, left_factor_k2, treat_epsilons_specially=False)
composed_query = k2.remove_epsilon_self_loops(composed_query)
composed_query = k2.connect(composed_query)
composed_query = k2.arc_sort(composed_query)
composed_query = k2.top_sort(composed_query)
composed_query = k2.prune_on_arc_post(k2.create_fsa_vec([composed_query]), 0.01, use_double_scores=True)
composed_query = k2.remove_epsilon_self_loops(composed_query)
composed_query.num_arcs

167

In [17]:
# lattice = k2.compose(composed_query.to("cpu"), k2.arc_sort(k2.add_epsilon_self_loops(lexicon_k2)).to("cpu"), treat_epsilons_specially=True)
# lattice = k2.create_fsa_vec([lattice])

In [20]:
lattice = k2.compose(composed_query, k2.add_epsilon_self_loops(lexicon_k2), treat_epsilons_specially=False)

OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB. GPU 0 has a total capacity of 23.51 GiB of which 1.79 GiB is free. Including non-PyTorch memory, this process has 21.57 GiB memory in use. Of the allocated memory 17.26 GiB is allocated by PyTorch, and 3.87 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [19]:
lattice = k2.remove_epsilon_self_loops(lattice)
lattice = k2.connect(lattice)
lattice = k2.arc_sort(lattice)
lattice = k2.top_sort(lattice)
nbest = Nbest.from_lattice(lattice, num_paths=3)
nbest.fsa.labels, extract_word_from_labels(nbest.fsa.labels, strip_eos=False)

(tensor([], device='cuda:0', dtype=torch.int32), '')